# 2024 SHED Survey Data Analysis

Survey of Household Economics and Decisionmaking - Federal Reserve

In [10]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Configure visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

In [11]:
# Load the optimized dataset
df = pd.read_csv('data_optimized.csv')

print(f"Dataset loaded: {len(df):,} respondents, {len(df.columns)} variables")
print(f"\nFirst few columns: {list(df.columns[:10])}")

Dataset loaded: 12,295 respondents, 385 variables

First few columns: ['CaseID', 'caseid2023', 'caseid2022', 'weight_pop', 'panel_weight_pop', 'L0_a', 'L0_b', 'L0_c', 'L0_d', 'L0_e']


/var/folders/kn/k2gq2hds1tg0q3tdmc6sxh4m0000gn/T/ipykernel_5521/2333796125.py:2: DtypeWarning: Columns (13,39,48,98,113,115,116,117,118,119,130,149,150,151,152,180,183,184,185,186,187,188,189,190,191,192,206,207,208,209,238,248,258,259,260,261,262,263,264,299,364,365,366,367) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data_optimized.csv')


---
## 1. Sample Composition

In [12]:
print("="*60)
print("SAMPLE COMPOSITION")
print("="*60)

# Total sample
print(f"\nTotal Respondents: {len(df):,}")

# Panel composition
panel_2023 = df['caseid2023'].notna().sum()
panel_2022 = df['caseid2022'].notna().sum()
panel_both = ((df['caseid2023'].notna()) & (df['caseid2022'].notna())).sum()

print(f"\n--- Panel Participation ---")
print(f"Also completed 2023 survey: {panel_2023:,} ({panel_2023/len(df)*100:.1f}%)")
print(f"Also completed 2022 survey: {panel_2022:,} ({panel_2022/len(df)*100:.1f}%)")
print(f"Completed all three years: {panel_both:,} ({panel_both/len(df)*100:.1f}%)")

# Demographics breakdown
print(f"\n--- Demographics ---")
print(f"\nGender:")
print(df['ppgender'].value_counts().to_string())

print(f"\nAge Categories:")
print(df['ppagecat'].value_counts().sort_index().to_string())

print(f"\nEducation:")
print(df['ppeducat'].value_counts().to_string())

print(f"\nRegion:")
print(df['ppreg4'].value_counts().to_string())

SAMPLE COMPOSITION

Total Respondents: 12,295

--- Panel Participation ---
Also completed 2023 survey: 4,039 (32.9%)
Also completed 2022 survey: 3,853 (31.3%)
Completed all three years: 2,077 (16.9%)

--- Demographics ---

Gender:
ppgender
Male      6230
Female    6065

Age Categories:
ppagecat
18-24     886
25-34    1799
35-44    2015
45-54    1774
55-64    2302
65-74    2219
75+      1300

Education:
ppeducat
Bachelor's degree or higher                                         5218
Some college or Associate's degree                                  3429
High school graduate (high school diploma or the equivalent GED)    2857
No high school diploma or GED                                        791

Region:
ppreg4
South        4575
West         2865
Midwest      2700
Northeast    2155


---
## 2. Financial Well-Being Overview

In [13]:
print("="*60)
print("FINANCIAL WELL-BEING OVERVIEW")
print("="*60)

# B2: Overall financial situation
print("\n--- Overall Financial Situation (B2) ---")
b2_counts = df['B2'].value_counts()
b2_pct = df['B2'].value_counts(normalize=True) * 100
print("\nHow are you managing financially?")
for category in b2_counts.index:
    if pd.notna(category):
        print(f"  {category}: {b2_counts[category]:,} ({b2_pct[category]:.1f}%)")

# Constructed variable: at least okay
print("\n--- Financial Well-Being Summary ---")
atleast_ok = df['atleast_okay'].value_counts()
if 'Public' in atleast_ok.index:
    pct_ok = atleast_ok['Public'] / atleast_ok.sum() * 100
    print(f"Doing at least okay financially: {atleast_ok['Public']:,} ({pct_ok:.1f}%)")

# $400 emergency expense
print("\n--- Emergency Savings ($400 Expense) ---")
pay_cash = df['pay_casheqv'].value_counts()
if 'Yes' in pay_cash.index:
    pct_cash = pay_cash['Yes'] / pay_cash.sum() * 100
    print(f"Can handle $400 with cash/equivalent: {pay_cash['Yes']:,} ({pct_cash:.1f}%)")

# B3: Financial situation vs. 12 months ago
print("\n--- Financial Trend (vs. 12 months ago) ---")
b3_counts = df['B3'].value_counts()
b3_pct = df['B3'].value_counts(normalize=True) * 100
for category in b3_counts.index:
    if pd.notna(category):
        print(f"  {category}: {b3_counts[category]:,} ({b3_pct[category]:.1f}%)")

# EF1-EF2: Emergency savings
print("\n--- Emergency Savings Capacity ---")
if 'EF1' in df.columns:
    print("\nHave set aside emergency/rainy day funds (EF1):")
    print(df['EF1'].value_counts().to_string())
    
if 'EF2' in df.columns:
    print("\nMonths of expenses covered (EF2):")
    print(df['EF2'].value_counts().to_string())

FINANCIAL WELL-BEING OVERVIEW

--- Overall Financial Situation (B2) ---

How are you managing financially?
  Doing okay: 4,730 (38.5%)
  Living comfortably: 4,391 (35.7%)
  Just getting by: 2,262 (18.4%)
  Finding it difficult to get by: 912 (7.4%)

--- Financial Well-Being Summary ---

--- Emergency Savings ($400 Expense) ---
Can handle $400 with cash/equivalent: 8,106 (65.9%)

--- Financial Trend (vs. 12 months ago) ---
  About the same: 5,934 (48.3%)
  Somewhat worse off: 2,746 (22.3%)
  Somewhat better off: 2,154 (17.5%)
  Much worse off: 790 (6.4%)
  Much better off: 671 (5.5%)

--- Emergency Savings Capacity ---

Have set aside emergency/rainy day funds (EF1):
EF1
Yes    7151
No     5144

Months of expenses covered (EF2):
EF2
No     3337
Yes    1807


---
## 3. Employment Snapshot

In [14]:
print("="*60)
print("EMPLOYMENT SNAPSHOT")
print("="*60)

# D1A: Work status
print("\n--- Work Status Last Month (D1A) ---")
d1a_counts = df['D1A'].value_counts()
d1a_pct = df['D1A'].value_counts(normalize=True) * 100
for category in d1a_counts.index:
    if pd.notna(category):
        print(f"  {category}: {d1a_counts[category]:,} ({d1a_pct[category]:.1f}%)")

# D3A: Employment type (for those working)
print("\n--- Employment Type (D3A) ---")
d3a_counts = df['D3A'].value_counts()
d3a_pct = df['D3A'].value_counts(normalize=True) * 100
for category in d3a_counts.index:
    if pd.notna(category):
        print(f"  {category}: {d3a_counts[category]:,} ({d3a_pct[category]:.1f}%)")

# Labor force participation
print("\n--- Labor Force Participation ---")
working = df['D1A'].isin(['Yes, full-time', 'Yes, part-time']).sum() if 'D1A' in df.columns else 0
total = df['D1A'].notna().sum()
if total > 0:
    lfp_rate = working / total * 100
    print(f"Labor Force Participation Rate: {lfp_rate:.1f}%")
    print(f"  Working: {working:,}")
    print(f"  Not working: {total - working:,}")

# D1I: Retired
if 'D1I' in df.columns:
    print("\n--- Retirement Status (D1I) ---")
    retired = df['D1I'].value_counts()
    print(retired.to_string())

# D44 series: Job transitions
print("\n--- Job Transitions in Past Year (D44 series) ---")
job_transitions = {
    'D44_a': 'Received a raise',
    'D44_b': 'Received a promotion',
    'D44_c': 'Took on a second job',
    'D44_d': 'Laid off or lost job',
    'D44_e': 'Started a new job',
    'D44_f': 'Voluntarily left job'
}

for var, label in job_transitions.items():
    if var in df.columns:
        yes_count = (df[var] == 'Yes').sum()
        total_responses = df[var].notna().sum()
        if total_responses > 0:
            pct = yes_count / total_responses * 100
            print(f"  {label}: {yes_count:,} ({pct:.1f}%)")

EMPLOYMENT SNAPSHOT

--- Work Status Last Month (D1A) ---
  Yes: 7,066 (57.5%)
  No: 5,229 (42.5%)

--- Employment Type (D3A) ---
  Working for someone else: 5,974 (84.5%)
  Self-employed (working for myself): 818 (11.6%)
  Other work arrangement: 274 (3.9%)

--- Labor Force Participation ---
Labor Force Participation Rate: 0.0%
  Working: 0
  Not working: 12,295

--- Retirement Status (D1I) ---
D1I
No     8247
Yes    4048

--- Job Transitions in Past Year (D44 series) ---
  Received a raise: 1,357 (19.2%)
  Received a promotion: 3,663 (51.8%)
  Took on a second job: 2,542 (20.7%)
  Laid off or lost job: 1,500 (12.2%)
  Started a new job: 963 (7.8%)
  Voluntarily left job: 714 (5.8%)


---
## 4. Housing & Costs

In [15]:
print("="*60)
print("HOUSING & COSTS")
print("="*60)

# GH1: Housing tenure
print("\n--- Housing Tenure (GH1) ---")
gh1_counts = df['GH1'].value_counts()
gh1_pct = df['GH1'].value_counts(normalize=True) * 100
for category in gh1_counts.index:
    if pd.notna(category):
        print(f"  {category}: {gh1_counts[category]:,} ({gh1_pct[category]:.1f}%)")

# R3: Monthly rent (for renters)
print("\n--- Monthly Rent (R3) ---")
if 'R3' in df.columns:
    rent_data = pd.to_numeric(df['R3'], errors='coerce')
    rent_valid = rent_data[rent_data.notna() & (rent_data > 0)]
    
    if len(rent_valid) > 0:
        print(f"  Renters reporting rent: {len(rent_valid):,}")
        print(f"  Median monthly rent: ${rent_valid.median():,.0f}")
        print(f"  Mean monthly rent: ${rent_valid.mean():,.0f}")
        print(f"  25th percentile: ${rent_valid.quantile(0.25):,.0f}")
        print(f"  75th percentile: ${rent_valid.quantile(0.75):,.0f}")

# M4: Monthly mortgage (for homeowners with mortgage)
print("\n--- Monthly Mortgage Payment (M4) ---")
if 'M4' in df.columns:
    mortgage_data = pd.to_numeric(df['M4'], errors='coerce')
    mortgage_valid = mortgage_data[mortgage_data.notna() & (mortgage_data > 0)]
    
    if len(mortgage_valid) > 0:
        print(f"  Homeowners reporting mortgage: {len(mortgage_valid):,}")
        print(f"  Median monthly mortgage: ${mortgage_valid.median():,.0f}")
        print(f"  Mean monthly mortgage: ${mortgage_valid.mean():,.0f}")
        print(f"  25th percentile: ${mortgage_valid.quantile(0.25):,.0f}")
        print(f"  75th percentile: ${mortgage_valid.quantile(0.75):,.0f}")

# GH3 series: Neighborhood satisfaction
print("\n--- Neighborhood Satisfaction (GH3 series) ---")
satisfaction_vars = {
    'GH3_a': 'Quality of public schools',
    'GH3_b': 'Safety from crime',
    'GH3_c': 'Quality of other amenities',
    'GH3_d': 'Overall quality of neighborhood',
    'GH3_e': 'Diversity of neighborhood'
}

for var, label in satisfaction_vars.items():
    if var in df.columns:
        satisfied = df[var].isin(['Very satisfied', 'Somewhat satisfied']).sum()
        total = df[var].notna().sum()
        if total > 0:
            pct = satisfied / total * 100
            print(f"  {label}: {pct:.1f}% satisfied")

HOUSING & COSTS

--- Housing Tenure (GH1) ---
  Own your home with a mortgage or loan: 5,078 (41.3%)
  Pay rent: 3,206 (26.1%)
  Own your home free and clear (without a mortgage or loan): 3,106 (25.3%)
  Neither own nor pay rent: 905 (7.4%)

--- Monthly Rent (R3) ---
  Renters reporting rent: 3,185
  Median monthly rent: $1,200
  Mean monthly rent: $1,379
  25th percentile: $730
  75th percentile: $1,750

--- Monthly Mortgage Payment (M4) ---
  Homeowners reporting mortgage: 4,977
  Median monthly mortgage: $1,500
  Mean monthly mortgage: $1,724
  25th percentile: $1,000
  75th percentile: $2,200

--- Neighborhood Satisfaction (GH3 series) ---
  Quality of public schools: 78.3% satisfied
  Safety from crime: 56.3% satisfied
  Quality of other amenities: 66.0% satisfied
  Overall quality of neighborhood: 68.5% satisfied
  Diversity of neighborhood: 38.6% satisfied


---
## 5. Income Distribution

In [16]:
print("="*60)
print("INCOME DISTRIBUTION")
print("="*60)

# I40: Total household income
print("\n--- Household Income Distribution (I40) ---")
if 'I40' in df.columns:
    i40_counts = df['I40'].value_counts()
    i40_pct = df['I40'].value_counts(normalize=True) * 100
    for category in i40_counts.index:
        if pd.notna(category):
            print(f"  {category}: {i40_counts[category]:,} ({i40_pct[category]:.1f}%)")

# Simplified income categories
print("\n--- Income Categories (inc_4cat_50k) ---")
if 'inc_4cat_50k' in df.columns:
    inc_cat_counts = df['inc_4cat_50k'].value_counts()
    inc_cat_pct = df['inc_4cat_50k'].value_counts(normalize=True) * 100
    for category in inc_cat_counts.index:
        if pd.notna(category):
            print(f"  {category}: {inc_cat_counts[category]:,} ({inc_cat_pct[category]:.1f}%)")

# I20: Spending relative to income
print("\n--- Spending vs. Income (I20) ---")
if 'I20' in df.columns:
    i20_counts = df['I20'].value_counts()
    i20_pct = df['I20'].value_counts(normalize=True) * 100
    for category in i20_counts.index:
        if pd.notna(category):
            print(f"  {category}: {i20_counts[category]:,} ({i20_pct[category]:.1f}%)")

# I0 series: Income sources
print("\n--- Income Sources (I0 series) ---")
income_sources = {
    'I0_a': 'Wages or salaries',
    'I0_b': 'Self-employment income',
    'I0_c': 'Social Security',
    'I0_d': 'Pension or retirement account',
    'I0_e': 'Interest/dividends/rental income',
    'I0_f': 'Other sources'
}

for var, label in income_sources.items():
    if var in df.columns:
        yes_count = (df[var] == 'Yes').sum()
        total = df[var].notna().sum()
        if total > 0:
            pct = yes_count / total * 100
            print(f"  {label}: {yes_count:,} ({pct:.1f}%)")

INCOME DISTRIBUTION

--- Household Income Distribution (I40) ---
  $100,000 to $149,999: 2,086 (17.0%)
  $75,000 to $99,999: 1,429 (11.6%)
  $200,000 or more: 1,395 (11.3%)
  $150,000 to $199,999: 1,330 (10.8%)
  $60,000 to $74,999: 1,070 (8.7%)
  $50,000 to $59,999: 871 (7.1%)
  $40,000 to $49,999: 726 (5.9%)
  Less than $5,000: 639 (5.2%)
  $35,000 to $39,999: 449 (3.7%)
  $30,000 to $34,999: 432 (3.5%)
  $10,000 to $14,999: 421 (3.4%)
  $20,000 to $24,999: 410 (3.3%)
  $25,000 to $29,999: 393 (3.2%)
  $15,000 to $19,999: 331 (2.7%)
  $5,000 to $9,999: 313 (2.5%)

--- Income Categories (inc_4cat_50k) ---
  $100,000 or more: 4,811 (39.1%)
  $50,000–$99,999: 3,370 (27.4%)
  Less than $25,000: 2,114 (17.2%)
  $25,000–$49,999: 2,000 (16.3%)

--- Spending vs. Income (I20) ---
  Less than your income: 6,315 (51.4%)
  The same as your income: 3,692 (30.0%)
  More than your income: 2,288 (18.6%)

--- Income Sources (I0 series) ---
  Wages or salaries: 7,952 (64.7%)
  Self-employment income: 

---
## 6. Key Demographic Breakdowns

In [17]:
print("="*60)
print("KEY DEMOGRAPHIC BREAKDOWNS")
print("="*60)

# Age distribution
print("\n--- Age Distribution ---")
if 'ppage' in df.columns:
    age_data = pd.to_numeric(df['ppage'], errors='coerce')
    age_valid = age_data[age_data.notna()]
    
    if len(age_valid) > 0:
        print(f"  Mean age: {age_valid.mean():.1f}")
        print(f"  Median age: {age_valid.median():.0f}")
        print(f"  Age range: {age_valid.min():.0f} - {age_valid.max():.0f}")

print("\n--- Age Categories (ppagecat) ---")
if 'ppagecat' in df.columns:
    age_cat_counts = df['ppagecat'].value_counts().sort_index()
    age_cat_pct = df['ppagecat'].value_counts(normalize=True).sort_index() * 100
    for category in age_cat_counts.index:
        if pd.notna(category):
            print(f"  {category}: {age_cat_counts[category]:,} ({age_cat_pct[category]:.1f}%)")

# Education
print("\n--- Education (educ_4cat) ---")
if 'educ_4cat' in df.columns:
    educ_counts = df['educ_4cat'].value_counts()
    educ_pct = df['educ_4cat'].value_counts(normalize=True) * 100
    for category in educ_counts.index:
        if pd.notna(category):
            print(f"  {category}: {educ_counts[category]:,} ({educ_pct[category]:.1f}%)")

# Race/ethnicity
print("\n--- Race/Ethnicity (race_5cat) ---")
if 'race_5cat' in df.columns:
    race_counts = df['race_5cat'].value_counts()
    race_pct = df['race_5cat'].value_counts(normalize=True) * 100
    for category in race_counts.index:
        if pd.notna(category):
            print(f"  {category}: {race_counts[category]:,} ({race_pct[category]:.1f}%)")

# Geographic distribution
print("\n--- Census Region (ppreg4) ---")
if 'ppreg4' in df.columns:
    region_counts = df['ppreg4'].value_counts()
    region_pct = df['ppreg4'].value_counts(normalize=True) * 100
    for category in region_counts.index:
        if pd.notna(category):
            print(f"  {category}: {region_counts[category]:,} ({region_pct[category]:.1f}%)")

# Household size
print("\n--- Household Size (pphhsize) ---")
if 'pphhsize' in df.columns:
    hhsize_counts = df['pphhsize'].value_counts().sort_index()
    hhsize_pct = df['pphhsize'].value_counts(normalize=True).sort_index() * 100
    for category in hhsize_counts.index:
        if pd.notna(category):
            print(f"  {category}: {hhsize_counts[category]:,} ({hhsize_pct[category]:.1f}%)")

KEY DEMOGRAPHIC BREAKDOWNS

--- Age Distribution ---
  Mean age: 51.6
  Median age: 53
  Age range: 18 - 97

--- Age Categories (ppagecat) ---
  18-24: 886 (7.2%)
  25-34: 1,799 (14.6%)
  35-44: 2,015 (16.4%)
  45-54: 1,774 (14.4%)
  55-64: 2,302 (18.7%)
  65-74: 2,219 (18.0%)
  75+: 1,300 (10.6%)

--- Education (educ_4cat) ---
  Bachelor's degree or more: 5,282 (43.0%)
  Some college/technical or associates degree: 4,135 (33.6%)
  High school degree or GED: 2,320 (18.9%)
  Less than a high school degree: 558 (4.5%)

--- Race/Ethnicity (race_5cat) ---
  White: 8,181 (66.5%)
  Hispanic: 1,685 (13.7%)
  Black: 1,377 (11.2%)
  Asian: 528 (4.3%)
  Other: 524 (4.3%)

--- Census Region (ppreg4) ---
  South: 4,575 (37.2%)
  West: 2,865 (23.3%)
  Midwest: 2,700 (22.0%)
  Northeast: 2,155 (17.5%)

--- Household Size (pphhsize) ---
  1: 2,284 (18.6%)
  2: 4,962 (40.4%)
  3: 2,143 (17.4%)
  4: 1,643 (13.4%)
  5: 708 (5.8%)
  6: 321 (2.6%)
  7: 125 (1.0%)
  8: 56 (0.5%)
  9: 24 (0.2%)
  10: 29 (0.

---
## 7. Financial Stress Indicators

In [18]:
print("="*60)
print("FINANCIAL STRESS INDICATORS")
print("="*60)

# Credit card behavior (C3P)
print("\n--- Credit Card Payment Behavior (C3P) ---")
if 'C3P' in df.columns:
    c3p_counts = df['C3P'].value_counts()
    c3p_pct = df['C3P'].value_counts(normalize=True) * 100
    for category in c3p_counts.index:
        if pd.notna(category):
            print(f"  {category}: {c3p_counts[category]:,} ({c3p_pct[category]:.1f}%)")

# Student loans
print("\n--- Student Loan Burden ---")
if 'SL1' in df.columns:
    print("Has student loan debt (SL1):")
    sl1_counts = df['SL1'].value_counts()
    sl1_pct = df['SL1'].value_counts(normalize=True) * 100
    for category in sl1_counts.index:
        if pd.notna(category):
            print(f"  {category}: {sl1_counts[category]:,} ({sl1_pct[category]:.1f}%)")

if 'SL6' in df.columns:
    print("\nBehind on student loan payments (SL6):")
    sl6_counts = df['SL6'].value_counts()
    sl6_pct = df['SL6'].value_counts(normalize=True) * 100
    for category in sl6_counts.index:
        if pd.notna(category):
            print(f"  {category}: {sl6_counts[category]:,} ({sl6_pct[category]:.1f}%)")

# Healthcare cost barriers (E1 series)
print("\n--- Healthcare Cost Barriers (E1 series) ---")
print("Went without the following due to cost:")
healthcare_barriers = {
    'E1_a': 'Prescription medicine',
    'E1_b': 'See a doctor/specialist',
    'E1_c': 'Mental health care/counseling',
    'E1_d': 'Dental care',
    'E1_e': 'Follow-up care'
}

for var, label in healthcare_barriers.items():
    if var in df.columns:
        yes_count = (df[var] == 'Yes').sum()
        total = df[var].notna().sum()
        if total > 0:
            pct = yes_count / total * 100
            print(f"  {label}: {yes_count:,} ({pct:.1f}%)")

# Inflation impacts (INF3 series)
print("\n--- Actions Taken Due to Price Increases (INF3 series) ---")
inflation_actions = {
    'INF3_a': 'Used savings',
    'INF3_b': 'Borrowed money or used credit',
    'INF3_c': 'Reduced spending on essentials',
    'INF3_d': 'Reduced spending on non-essentials',
    'INF3_e': 'Worked more/additional job',
    'INF3_f': 'Asked for raise/changed jobs',
    'INF3_g': 'Other actions'
}

for var, label in inflation_actions.items():
    if var in df.columns:
        yes_count = (df[var] == 'Yes').sum()
        total = df[var].notna().sum()
        if total > 0:
            pct = yes_count / total * 100
            print(f"  {label}: {yes_count:,} ({pct:.1f}%)")

# Credit application denials
print("\n--- Credit Access ---")
if 'A0' in df.columns:
    print("Applied for credit in past 12 months (A0):")
    a0_counts = df['A0'].value_counts()
    for category in a0_counts.index:
        if pd.notna(category):
            print(f"  {category}: {a0_counts[category]:,}")

FINANCIAL STRESS INDICATORS

--- Credit Card Payment Behavior (C3P) ---
  paid at least the minimum payment on all credit cards: 9,463 (91.4%)
  did not use any of my credit cards so had no balances: 627 (6.1%)
  did not pay or paid less than the minimum payment on at least one card: 259 (2.5%)

--- Student Loan Burden ---
Has student loan debt (SL1):
  No: 10,828 (88.1%)
  Yes: 1,467 (11.9%)

Behind on student loan payments (SL6):
  No: 1,177 (80.2%)
  Yes: 290 (19.8%)

--- Healthcare Cost Barriers (E1 series) ---
Went without the following due to cost:
  Prescription medicine: 1,246 (10.1%)
  See a doctor/specialist: 1,845 (15.0%)
  Mental health care/counseling: 1,079 (8.8%)
  Dental care: 2,306 (18.8%)
  Follow-up care: 1,211 (9.8%)

--- Actions Taken Due to Price Increases (INF3 series) ---
  Used savings: 7,720 (62.8%)
  Borrowed money or used credit: 7,487 (60.9%)
  Reduced spending on essentials: 5,206 (42.3%)
  Reduced spending on non-essentials: 1,804 (14.7%)
  Worked more/ad